In [106]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
import json

In [107]:
path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

X_train_df = pd.read_csv(train_path)
X_test_df = pd.read_csv(test_path)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

Train: (102984, 69)
Test : (25984, 69)


In [108]:
X_train_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_test = X_test_df[X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy() # to nasz zbiór testowy

In [109]:
acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]

In [110]:
X_train_source_acc = X_train_source[acc_features].fillna(0)
X_train_source_mic = X_train_source[mic_features].fillna(0)
X_train_source_gyro = X_train_source[gyro_features].fillna(0)


X_test_acc = X_test[acc_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)

In [111]:
def run_isolation_forest(X_train, X_test, test_df, sensor_type, params):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = IsolationForest(
        n_estimators=params["n_estimators"],
        max_samples=params["max_samples"],
        max_features=params["max_features"],
        bootstrap=params["bootstrap"],
        contamination="auto",
        random_state=42,
        n_jobs=-1
    )
    agg_method = params["aggregation"]
    threshold = params["threshold"]
    
    model.fit(X_train_scaled)

    test_scores = -model.score_samples(X_test_scaled)
    
    results = pd.DataFrame({"segment_id": test_df["segment_id"].values,
        "anomaly_label": test_df["anomaly_label"].values,
        "score": test_scores})
    
    agg_functions = {
        "max": "max",
        "mean": "mean",
        "median": "median"
    }
    segment_scores = (results.groupby("segment_id").agg(score=("score", agg_functions[agg_method]), 
                                                        anomaly_label=("anomaly_label", "first")).reset_index())
    
    segment_scores["y_true"] = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)
    segment_scores["y_pred"] = (segment_scores["score"] >= threshold).astype(int)

    y_true = segment_scores["y_true"]
    y_pred = segment_scores["y_pred"]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    roc_auc = roc_auc_score(y_true, segment_scores["score"]) 

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"{sensor_type}")
    print(f"Threshold: {threshold:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")
    print(cm)

    metrics = {"Sensors": sensor_type,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]}

    return metrics, segment_scores, model

In [112]:
params_path = path / "final_if_params.json"

with open(params_path, "r") as f:
    final_params = json.load(f)

print("final parameters:")
print(json.dumps(final_params, indent=4))

final parameters:
{
    "ACC": {
        "n_estimators": 50,
        "max_samples": 1024,
        "max_features": 0.8,
        "bootstrap": false,
        "threshold": 0.5268351498152332,
        "aggregation": "max"
    },
    "MIC": {
        "n_estimators": 50,
        "max_samples": 2048,
        "max_features": 0.9,
        "bootstrap": false,
        "threshold": 0.3660668015568886,
        "aggregation": "median"
    },
    "GYRO": {
        "n_estimators": 50,
        "max_samples": 2048,
        "max_features": 0.8,
        "bootstrap": false,
        "threshold": 0.3972321404293511,
        "aggregation": "median"
    }
}


In [113]:
params_acc = final_params["ACC"]
params_mic = final_params["MIC"]
params_gyro = final_params["GYRO"]

In [114]:
metrics_acc, scores_acc, model_acc = run_isolation_forest(X_train_source_acc, X_test_acc, X_test, "acc", params_acc)
metrics_mic, scores_mic, model_mic = run_isolation_forest(X_train_source_mic, X_test_mic, X_test, "mic", params_mic)
metrics_gyro, scores_gyro, model_gyro = run_isolation_forest(X_train_source_gyro, X_test_gyro, X_test, "gyro", params_gyro)

acc
Threshold: 0.5268
Accuracy : 0.5474
Precision: 0.5314
Recall   : 0.8017
F1-score : 0.6392
ROC AUC  : 0.4169
[[34 82]
 [23 93]]
mic
Threshold: 0.3661
Accuracy : 0.5474
Precision: 0.5261
Recall   : 0.9569
F1-score : 0.6789
ROC AUC  : 0.5887
[[ 16 100]
 [  5 111]]
gyro
Threshold: 0.3972
Accuracy : 0.5991
Precision: 0.5550
Recall   : 1.0000
F1-score : 0.7138
ROC AUC  : 0.6108
[[ 23  93]
 [  0 116]]


segment jest klasyfikowany jako anomalia, jeśli co najmniej dwa z trzech modeli wykryją anomalię

In [115]:
final_results = (scores_acc[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
    .merge(scores_mic[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
    .merge(scores_gyro[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))
final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results (majority voting):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)

Final Results (majority voting):
Accuracy : 0.5776
Precision: 0.5425
Recall   : 0.9914
F1-score : 0.7012
Confusion matrix:
[[ 19  97]
 [  1 115]]
